In [0]:
%pip install --upgrade groq langchain-groq

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import getpass

# This will pop up a text box below the cell. Paste your key and hit Enter.
# The characters will be hidden (e.g., ••••••••)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass(prompt="Enter your Groq API Key: ")

print("✅ Key securely loaded into environment variables.")

Enter your Groq API Key:  [REDACTED]

✅ Key securely loaded into environment variables.


In [0]:
import os
from langchain_groq import ChatGroq

# 1. Quick safety check to ensure our API key is still in memory
if "GROQ_API_KEY" not in os.environ:
    print("⚠️ API Key missing! Please scroll up and run the 'getpass' cell from Sprint 1 again.")
else:
    # 2. Initialize the brain (The LLM)
    llm = ChatGroq(
        # Using the current, active Groq model
        model="llama-3.1-8b-instant", 
        temperature=0,
        max_retries=2
    )
    print("✅ Brain (LLM) successfully initialized! The Planner Node is ready.")

✅ Brain (LLM) successfully initialized! The Planner Node is ready.


In [0]:
# (The cell with this code)
final_state = spark_sensei.invoke(initial_state)

🧠 [Planner Node]: Thinking and generating SQL...
⚙️ [Executor Node]: Running SQL: 
🚦 [Router]: Success detected. Stopping graph.


In [0]:
# View the first 5 rows of our target dataset
display(spark.sql("SELECT * FROM samples.nyctaxi.trips LIMIT 5"))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022


In [0]:
from langchain_core.tools import tool

@tool
def get_table_schema(table_name: str) -> str:
    """
    Always use this tool to get the exact schema of a table before writing any SQL query.
    Input should be the fully qualified table name (e.g., 'samples.nyctaxi.trips').
    Returns a string with column names and their data types.
    """
    # Defensive Check: Does this table actually exist?
    if not spark.catalog.tableExists(table_name):
        return f"Error: Table '{table_name}' not found. Did you spell it correctly?"
        
    try:
        # Fetch the catalog metadata from Spark
        columns = spark.catalog.listColumns(table_name)
        
        # Format it into a clean Markdown-style string for the LLM
        schema_str = f"Schema for table '{table_name}':\n"
        for col in columns:
            schema_str += f"- {col.name} ({col.dataType})\n"
            
        return schema_str
        
    except Exception as e:
        # If it fails for another weird reason, return a hardcoded short string, NOT the Java error
        return f"Error: Could not read schema for '{table_name}'. It may be corrupted or inaccessible."
    
@tool
def list_tables_in_db(database_name: str) -> str:
    """
    Use this tool to find out what tables exist inside a specific database.
    Input should be the database name (e.g., 'samples.nyctaxi').
    Returns a list of all table names in that database.
    """
    try:
        # Fetch the list of tables from the Spark catalog
        tables = spark.catalog.listTables(database_name)
        
        if not tables:
            return f"No tables found in database '{database_name}'."
            
        table_list = f"Tables in '{database_name}':\n"
        for t in tables:
            table_list += f"- {t.name}\n"
            
        return table_list
    except Exception as e:
        return f"Error reading database '{database_name}': {str(e)}"

In [0]:
# Test 1: A valid table
print("--- Test 1: Valid Table ---")
print(get_table_schema.invoke("samples.nyctaxi.trips"))

# Test 2: A table that doesn't exist to ensure our error handling works
print("\n--- Test 2: Invalid Table ---")
print(get_table_schema.invoke("samples.nyctaxi.fake_table"))

--- Test 1: Valid Table ---
Schema for table 'samples.nyctaxi.trips':
- tpep_pickup_datetime (timestamp)
- tpep_dropoff_datetime (timestamp)
- trip_distance (double)
- fare_amount (double)
- pickup_zip (int)
- dropoff_zip (int)


--- Test 2: Invalid Table ---
Error: Table 'samples.nyctaxi.fake_table' not found. Did you spell it correctly?


In [0]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

# This defines the "backpack" our agent carries between nodes
class AgentState(TypedDict):
    # 'messages' stores the conversation history. 
    # add_messages tells LangGraph to append new messages, not overwrite old ones.
    messages: Annotated[list[AnyMessage], add_messages]
    
    # The generated SQL query
    sql_query: str
    
    # The error message if Spark fails
    error_message: str
    
    # How many times the agent has tried to fix an error
    retry_count: int

In [0]:
import json
from langchain_core.messages import SystemMessage, AIMessage, ToolMessage

# --- 1. THE CUSTOM TOOL WORKSTATION ---
def tools_node(state: AgentState):
    print("🛠️ [Tool Node]: Executing requested tools...")
    last_message = state["messages"][-1]
    
    tool_responses = []
    # Loop through all the tools the AI asked to use
    # Inside your tools_node function:
    for tool_call in last_message.tool_calls:
        if tool_call["name"] == "get_table_schema":
            table_name = tool_call["args"].get("table_name")
            result = get_table_schema.invoke(table_name)
            
        # --- NEW CODE BELOW ---
        elif tool_call["name"] == "list_tables_in_db":
            db_name = tool_call["args"].get("database_name")
            print(f"   -> Scanning database: {db_name}")
            result = list_tables_in_db.invoke(db_name)
        # --- NEW CODE ABOVE ---
        
        tool_responses.append(
            ToolMessage(content=str(result), name=tool_call["name"], tool_call_id=tool_call["id"])
        )
            
    return {"messages": tool_responses}

# --- 2. THE PLANNER NODE ---
def planner_node(state: AgentState):
    print("🧠 [Planner Node]: Thinking...")
    llm_with_tools = llm.bind_tools([get_table_schema, list_tables_in_db])
    
    system_prompt = SystemMessage(content="""
    You are a PySpark SQL expert on Databricks.
    1. ALWAYS use the 'get_table_schema' tool to check the table schema before writing SQL.
    2. Once you have the schema, write the PySpark SQL query.
    3. Output ONLY the raw SQL query. Do not wrap it in ```sql ``` markdown blocks.
    """)
    
    messages_to_send = [system_prompt] + state["messages"]
    response = llm_with_tools.invoke(messages_to_send)
    
    # Extract SQL ONLY if the LLM didn't ask for a tool
    clean_sql = ""
    if not response.tool_calls and response.content:
        clean_sql = response.content.replace("```sql", "").replace("```", "").strip()
        
    return {"messages": [response], "sql_query": clean_sql}

# --- 3. THE EXECUTOR NODE ---
def executor_node(state: AgentState):
    query = state.get("sql_query", "")
    print(f"⚙️ [Executor Node]: Running SQL:\n{query}")
    
    if not query:
        return {"error_message": "LLM failed to generate a SQL query."}
        
    try:
        df = spark.sql(query)
        result_str = df.limit(5).toPandas().to_string()
        print("✅ [Executor Node]: Success!")
        return {"error_message": "", "messages": [AIMessage(content=f"Query succeeded. Result preview:\n{result_str}")]}
    except Exception as e:
        print(f"❌ [Executor Node]: Spark Error Caught!")
        return {"error_message": str(e)}

In [0]:
from langgraph.graph import StateGraph, END

def route_after_planner(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        print("🚦 [Router]: AI requested a tool. Routing to Tool Node.")
        return "tools"
    print("🚦 [Router]: AI generated SQL. Routing to Executor.")
    return "executor"

def should_heal(state: AgentState):
    error = state.get("error_message")
    retries = state.get("retry_count", 0)
    
    if error and retries < 3:
        print(f"🚦 [Router]: Error detected. Routing to Healer (Retry {retries + 1}/3)")
        return "heal"
    elif error and retries >= 3:
        print("🚦 [Router]: Max retries reached.")
        return "end"
    else:
        print("🚦 [Router]: Success detected. Stopping graph.")
        return "end"

print("🏗️ Re-Building the LangGraph...")
workflow = StateGraph(AgentState)

# Add our custom nodes
workflow.add_node("planner", planner_node)
workflow.add_node("tools", tools_node) # Using our custom function now!
workflow.add_node("executor", executor_node)
workflow.add_node("healer", healer_node)

workflow.set_entry_point("planner")

workflow.add_conditional_edges("planner", route_after_planner, {"tools": "tools", "executor": "executor"})
workflow.add_edge("tools", "planner") 
workflow.add_conditional_edges("executor", should_heal, {"heal": "healer", "end": END})
workflow.add_edge("healer", "planner")

spark_sensei = workflow.compile()
print("✅ Spark-Sensei Graph Re-Compiled Successfully!")

🏗️ Re-Building the LangGraph...
✅ Spark-Sensei Graph Re-Compiled Successfully!


In [0]:
from langchain_core.messages import HumanMessage

# 1. The Question
user_question = """
Look at the 'samples.nyctaxi' database. 
Find out what tables exist,then display the top 5 records on each table.
"""

# 2. THE CRITICAL STEP: Completely overwrite the old backpack!
initial_state = {
    "messages": [HumanMessage(content=user_question)],
    "sql_query": "",
    "error_message": "",
    "retry_count": 0
}

# 3. Invoke the graph
print("🚀 Starting Project Spark-Sensei with fresh memory...\n")
final_state = spark_sensei.invoke(initial_state)

print("\n🏁 FINAL RESULT:")
print(final_state["messages"][-1].content)

🚀 Starting Project Spark-Sensei with fresh memory...

🧠 [Planner Node]: Thinking...
🚦 [Router]: AI requested a tool. Routing to Tool Node.
🛠️ [Tool Node]: Executing requested tools...
   -> Scanning database: samples.nyctaxi
   -> Scanning database: samples.nyctaxi
🧠 [Planner Node]: Thinking...
🚦 [Router]: AI requested a tool. Routing to Tool Node.
🛠️ [Tool Node]: Executing requested tools...
🧠 [Planner Node]: Thinking...
🚦 [Router]: AI generated SQL. Routing to Executor.
⚙️ [Executor Node]: Running SQL:
SELECT * FROM samples.nyctaxi.trips LIMIT 5
✅ [Executor Node]: Success!
🚦 [Router]: Success detected. Stopping graph.

🏁 FINAL RESULT:
Query succeeded. Result preview:
  tpep_pickup_datetime tpep_dropoff_datetime  trip_distance  fare_amount  pickup_zip  dropoff_zip
0  2016-02-13 21:47:53   2016-02-13 21:57:15           1.40          8.0       10103        10110
1  2016-02-13 18:29:09   2016-02-13 18:37:23           1.31          7.5       10023        10023
2  2016-02-06 19:40:58   201

In [0]:
initial_state

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can